# QEvict completion ablation for Reviewer 2: LongBench long-generation tasks, batched (Colab, A100)

> *How sensitive is QEvict's model quality to the accuracy of the calibrated window-level
> completion term for windows that are not selected for token-level attention?*

**Model:** Llama-3.1-8B-Instruct. **Cache:** budget **0.10**, W = 8, `quant_ratio` 0.70 in `bytes` mode, int2, flash backend,
`bidir`/`dequant`, greedy. Read ratio ρ = 0.25 (the shipped gate), with ρ = 0.10 as the stress point and ρ = 1.0 as the
control.

**Tasks.** LongBench's three English tasks with 512-token generations: `gov_report`, `qmsum` and `multi_news`.
* **50 examples each.** These are the first 50 examples of each task long enough for its fixed prompt length.
* **Fixed prompt lengths.** Every prompt is middle-truncated to exactly `PROMPT_TOKENS[task]` tokens: 6,144 for
  `gov_report` and `qmsum` (median full length 10.6k and 13.2k), and 2,048 for `multi_news` (median 1.9k). The truncation
  follows THUDM's rule, applied to the final token ids.
* **Why.** The windowed cache batches only rows of exactly equal length, because it takes no padding. With one length per
  task, a task's 50 rows decode together (`BATCH_SIZE`, capped by `BATCH_TOKENS` for memory). The prompts are the same in
  every arm, so every comparison stays paired.

| Experiment | Question | Where |
|---|---|---|
| **1** Matched-state accuracy | How close is the completion mass to the true unread mass? How much attention-output error does each variant cause **on the same decode state**, at ρ = 0.25 and ρ = 0.10? | §6: the collector (4 prompts per task and ρ). §7: `rec_*` arms record every variant inside the real batched runs, on all 50 prompts per task |
| **2** Is completion needed? | `cal` vs `off`; the ρ = 1.0 ceiling; the ρ = 0.10 stress pair, with its difference-in-differences | LongBench |
| **3** Is calibration needed? | `cal` vs `uncal` | LongBench |
| **4** How sensitive is quality? | Common scale c ∈ {0.5, 0.8, 1.25, 2, 4}; total-preserving noise at 1×, 2× and 4× σ̂_U over seeds 0, 1 and 2; coupled scores | LongBench |

`cal` is the shipped method and the control in every comparison. Two floors run on all 50 examples per task:
* `cal_repeat` is a second run of the same configuration;
* `cal_ep` sends the shipped term through the evaluation epilogue at identity.

The two recorder arms are `cal` and `stress_g0.10_cal` with recording on. Their predictions are scored as well, and must
match their unrecorded twins to within the floors: recording only reads.

### The knobs
* **Completion** (`utils/completion_spec.py`): `cache.quant_completion`. With `null`, the shipped kernel runs. With any
  other value, the evaluation epilogue (`modules/windowed_cache/completion.py`) recovers the kernel's terms exactly and
  applies the variant after selection. Use `off`, not `none`.
* **Batching** (`longbench_runner._run_dataset_batched`): `longbench.prompt_tokens.<task>`, `batch_size` and
  `batch_tokens`. Each row is cut at its own EOS, and a batch that runs out of memory is halved. The sidecar's `batching`
  block records the plan, the realised batches and the truncation.
* **Recording** (`modules/evaluation/completion_recorder.py`): `longbench.completion_rungs` and its `_stride`. Every
  stride-th decode step, every layer, every live row: all 15 completion variants on the state the step just read. The
  data are reduced on the GPU to per-head statistics.

### Why a number can be trusted
Each sidecar must carry all of the following, or its result is moved to `refused/` and not scored:
* `read_gate.verdict == "gated"`;
* the realised read fraction for the requested ρ;
* the operating point, at budget 0.10;
* `quant_completion` equal to the arm's spec, with its runtime proof: `completion_applied == gated` on variant arms and 0
  on native ones;
* the task's fixed prompt length;
* all 50 examples present;
* matched-state records on exactly the recorder arms.

The matched-state data are refused unless the recomputed calibrated output reproduces the kernel's, and the full oracle
reproduces `o_held`.

One Drive folder holds one commit, and this run has its own folder, so nothing from earlier runs is read or rewritten.

### Time (one A100 80 GB; arithmetic, re-projected from measured speed after the first arm)
| Stage | What | Estimate |
|---|---|---|
| Setup | venv (first time ~8 min), model download, preflight tests | ~15 min |
| §6 Experiment 1 | collector, 3 tasks × 2 ρ × 4 prompts × 128 steps | ~15–25 min |
| §7 LongBench | 26 arms × 150 examples: per task 1–2 batches of 25–50 rows, ≤ 512 steps each | ~2.5–3.5 h |
| §8–9 | scoring, analysis, figures | minutes |

The total is about 3.5–4.5 h. The arms run in priority order, and finished arms are kept on Drive, so a run cut short
still has the comparisons that matter most.

### How to use
1. Runtime → **A100** (High-RAM helps). Colab secrets: `HF_TOKEN` (Llama-3.1 access), and `GITHUB_TOKEN` if the repo is
   private.
2. Run top to bottom. After a disconnect, run everything again: finished arms are skipped.
3. Section 4 prints the batch plan per task before anything long runs: examples skipped as too short, tokens truncated,
   and rows per batch.

## 0. Settings

In [ ]:
# ===================== Settings: edit here =====================
BRANCH       = "Clustered_int2_paused_fast"
REPO_URL     = "https://github.com/shaswata965-coder/RecoverKv.git"
PIN_COMMIT   = "021fde525bb56bf6882a882b40207d9ac51dad06"   # the commit with batched LongBench and the recorder
# Every arm runs on this commit. Quality rows do not compare across commits. Change it only together with a new DRIVE_ROOT.
MODEL_ID     = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_DIR    = "/content/models/Llama-3.1-8B-Instruct"   # the name must contain "instruct": the chat-template gate reads it
DRIVE_ROOT   = "/content/drive/MyDrive/recoverkv_completion_ablation_b010_longgen"   # this run's own folder

TASKS         = ["gov_report", "qmsum", "multi_news"]   # LongBench's English tasks with 512-token generations
PROMPT_TOKENS = {"gov_report": 6144, "qmsum": 6144, "multi_news": 2048}   # every prompt: exactly this many tokens
NUM_SAMPLES        = 50     # per task, every arm: the first 50 examples at least PROMPT_TOKENS long
NUM_SAMPLES_FLOOR  = 50     # cal_repeat and cal_ep
CACHE_BUDGET       = 0.10
BATCH_SIZE         = 50     # rows decoded together, capped so that rows x prompt tokens <= BATCH_TOKENS
BATCH_TOKENS       = None   # None: 160k on an 80 GB GPU, 48k on 40 GB (set in section 1)
RUN                = "all"  # "core" | "extended" (adds ceiling, stress, recorder arms, scales, noise) | "all" (+ noise seeds 1, 2)
NOISE_MULTS        = (1, 2, 4)          # sigma = mult x sigma_hat_U
NOISE_SEEDS        = (0, 1, 2)
RECORD_STRIDE      = 8      # recorder arms: matched-state record every 8th decode step (64 per 512-token row)

# Experiment 1 (the collector): the shipped method, every variant computed on the same states, at both read ratios
EXP1_GATES   = (0.25, 0.10)
EXP1_PROMPTS = 4            # documents per task and ratio
EXP1_PREFILL = dict(PROMPT_TOKENS)   # the first N tokens of each document's text
EXP1_GEN     = 128
EXP1_CAL_VS_KERNEL_MAX = 1e-2        # refuse matched-state data whose recomputed calibrated output strays this far (median)

BOOTSTRAP_ITERS        = 10_000
RUN_PREFLIGHT_TESTS    = True
FLASH_ATTN_FROM_SOURCE = False   # only if no prebuilt wheel matches (slow: ~1-2 h)
ALLOW_NON_A100         = False
PAPER_ERROR_REDUCTION  = None    # e.g. 0.58: the matched-state figure quoted to the reviewer; section 9 checks it

# ===================== Fixed: these define the experiment =====================
OPERATING_POINT = dict(cache_budget=CACHE_BUDGET, window_size=8, quant_ratio=0.70,
                       quant_budget_mode="bytes", quant_promotion="bidir",
                       quant_promote_source="dequant")
GATE = 0.25
STRESS_GATE = 0.10
STAGES = {"core": ("core",), "extended": ("core", "extended", "collect"),
          "all": ("core", "extended", "collect", "followup")}
assert RUN in STAGES, RUN
assert set(PROMPT_TOKENS) >= set(TASKS), "every task needs a fixed prompt length"

## 1. GPU, Drive, repository

In [ ]:
import subprocess, json, os, sys, shutil, time, math, glob, zipfile, io, re, warnings, difflib
from pathlib import Path

q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                    "--format=csv,noheader,nounits"], capture_output=True, text=True)
if q.returncode != 0:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> A100.")
gpu_name, mem_mib, cc = [x.strip() for x in q.stdout.strip().splitlines()[0].split(",")]
GPU_MEM_GB = float(mem_mib) / 1024
print(f"GPU: {gpu_name}  {GPU_MEM_GB:.0f} GB  sm_{cc.replace('.', '')}")
if cc != "8.0" and not ALLOW_NON_A100:
    raise RuntimeError(f"Expected an A100 (sm_80), got {gpu_name} (sm_{cc}). The kernels were "
                       "compile-checked for sm80. Set ALLOW_NON_A100 = True to override.")
SKIP_OOM = GPU_MEM_GB < 70
if BATCH_TOKENS is None:                  # rows x prompt tokens per batch: what fits at prefill
    BATCH_TOKENS = 160_000 if GPU_MEM_GB >= 70 else 48_000
print(f"batch token budget: {BATCH_TOKENS:,} (rows x prompt tokens per batch)")
print(f"skip_oom = {SKIP_OOM}" + ("  (40 GB card: an OOM example scores 0 THUDM-style and is dropped "
      "from paired comparisons in every arm)" if SKIP_OOM else ""))

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
ROOT = Path(DRIVE_ROOT)
for d in ("longbench", "gsm8k", "exp1", "logs", "refused", "results"):
    (ROOT / d).mkdir(parents=True, exist_ok=True)

def _secret(k):
    try:
        return userdata.get(k)
    except Exception:
        return None

REPO = Path("/content/RecoverKv")
gh = _secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://x-access-token:{gh}@") if gh else REPO_URL

def _git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:  # never echo the tokenised URL
        raise RuntimeError("git " + args[0] + " failed:\n" + r.stderr.replace(gh or "\0", "***"))
    return r.stdout.strip()

if not REPO.exists():
    _git("clone", "--branch", BRANCH, url, str(REPO))
else:
    _git("fetch", "origin", BRANCH, cwd=REPO)
    _git("checkout", "-B", BRANCH, f"origin/{BRANCH}", cwd=REPO)
if PIN_COMMIT:
    _git("checkout", PIN_COMMIT, cwd=REPO)
SHA = _git("rev-parse", "HEAD", cwd=REPO)
print("commit", SHA, "(pinned)" if PIN_COMMIT else "  <- set PIN_COMMIT to this before any rerun")

# Quality rows do not compare across commits, so one Drive root holds one commit.
manifest = ROOT / "manifest.json"
if manifest.exists():
    m = json.loads(manifest.read_text())
    if m["sha"] != SHA:
        raise RuntimeError(f"{ROOT} holds results from commit {m['sha'][:10]}, and this clone is "
                           f"{SHA[:10]}. Set PIN_COMMIT = '{m['sha'][:10]}' to resume, or point "
                           "DRIVE_ROOT at a new folder.")
else:
    manifest.write_text(json.dumps({"sha": SHA, "branch": BRANCH, "model": MODEL_ID,
                                    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
                                   indent=2))

## 2. Install the pinned stack

Colab's kernel is Python 3.13 with its own, newer torch. The pinned stack goes into a separate
**Python 3.12 venv** (`/content/py312_env`), and every job below runs with that interpreter (`PY`).
The notebook kernel never imports torch, so no runtime restart is needed.

* torch 2.6.0 is the build the recorded GPU runs used. Do not move it: on torch 2.7.1 the eviction's
  `torch.compile` fails at decode step 0 (`StarDep does not have an index`), which stops every arm.
* transformers is pinned to 4.47.1: the windowed cache refuses anything newer.
* `datasets` stays below 3, as `environment.yml` pins it.
* flash-attn comes from a **prebuilt wheel**, installed with `--no-deps`, and is kept only if its CUDA
  extension actually imports against the venv's torch. In practice that is **2.7.4.post1**: the 2.8.x wheels
  labelled `cu12torch2.6cxx11abiFALSE` are built with the new C++ ABI and fail against pip's torch 2.6.0
  with `undefined symbol: _ZN3c105ErrorC2ENS_14SourceLocationENSt7__cxx11...` (checked: torch 2.6.0 exports
  that constructor only in the old-ABI form).
* Every install after torch runs under a constraints file (`torch==2.6.0`, `triton==3.2.0`, ...), so nothing
  can move torch.

**Why the previous version of this cell failed** (`undefined symbol:
_ZN3c104cuda29c10_cuda_check_implementationEiPKcS2_ib`): it installed flash-attn with `--force-reinstall`.
That flag reinstalls the package's dependencies too, and flash-attn depends on an unpinned `torch`, so pip
replaced torch 2.6.0 with the newest torch on PyPI. The flash-attn binary had been built against torch 2.6's
headers, and the newer libtorch no longer exports that symbol in that form. Had torch stayed at 2.6.0, the
2.8.3 wheel would still have failed, on the ABI symbol above. The cell now pins torch with a constraints
file, checks the installed versions before anything imports flash-attn, and tests the import itself.

If no wheel matches, set `FLASH_ATTN_FROM_SOURCE = True` (about 1–2 h, under the same constraints).

In [ ]:
import importlib.util

VENV = Path("/content/py312_env")
PY = str(VENV / "bin" / "python")          # every job runs with this interpreter
if sys.executable.startswith(str(VENV)):   # an older version of this cell pointed the kernel at the venv
    sys.executable = getattr(sys, "_base_executable", None) or "/usr/bin/python3"
KERNEL_PY = sys.executable                 # the notebook kernel itself: runs only the scoring and plots

PINS = {"torch": "2.6.0", "torchvision": "0.21.0", "torchaudio": "2.6.0", "triton": "3.2.0",
        "transformers": "4.47.1"}
STACK_ID = json.dumps({"python": "3.12", "cuda": "cu124", **PINS}, sort_keys=True)
MARK = VENV / ".recoverkv_stack"           # written only after the environment check below passes

# Nothing from Colab's own Python may leak into the venv, at install time or at run time.
_DROP = ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "CPATH", "C_INCLUDE_PATH", "CPLUS_INCLUDE_PATH",
         "PIP_CONSTRAINT", "MPLBACKEND")
VENV_ENV = {k: v for k, v in os.environ.items() if k not in _DROP}
VENV_ENV.update(VIRTUAL_ENV=str(VENV), PATH=f"{VENV / 'bin'}:{os.environ.get('PATH', '')}",
                MPLBACKEND="Agg")   # scripts that call plain `python` get the venv too

def sh(*cmd, env=None):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd)[:300])
    subprocess.run(cmd, check=True, env=VENV_ENV if env is None else env)

def vpip(*a):
    sh(PY, "-m", "pip", "install", "-q", "--disable-pip-version-check", *a)

def venv_versions():
    code = ("import json, importlib.metadata as m\nout = {}\n"
            "for d in ('torch','torchvision','torchaudio','triton','transformers','flash_attn','datasets','numpy'):\n"
            "    try: out[d] = m.version(d)\n    except m.PackageNotFoundError: out[d] = None\n"
            "print(json.dumps(out))")
    return json.loads(subprocess.check_output([PY, "-c", code], env=VENV_ENV, text=True))

def check_pins():
    v = venv_versions()
    bad = {k: v.get(k) for k, want in PINS.items() if (v.get(k) or "").split("+")[0] != want}
    if bad:
        raise RuntimeError(f"the venv drifted from the pinned stack: {bad}, want {PINS}. Something "
                           "reinstalled torch (e.g. a --force-reinstall, or an unpinned dependency). "
                           f"Delete {VENV} and rerun this cell.")
    return v

if not (MARK.exists() and MARK.read_text() == STACK_ID and Path(PY).exists()):
    if VENV.exists():
        print(f"removing {VENV}: not built by this cell, or for a different stack")
        shutil.rmtree(VENV)
    has312 = shutil.which("python3.12") and subprocess.run(
        ["python3.12", "-c", "import ensurepip, venv"], capture_output=True).returncode == 0
    if not has312 or FLASH_ATTN_FROM_SOURCE:  # a source build also needs the Python headers
        apt_env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
        sh("apt-get", "update", "-qq", env=apt_env)
        sh("apt-get", "install", "-y", "-qq", "python3.12", "python3.12-venv", "python3.12-dev", env=apt_env)
    sh("python3.12", "-m", "venv", VENV, env=os.environ)
    constraints = VENV / "constraints.txt"
    constraints.write_text("".join(f"{k}=={v}\n" for k, v in PINS.items()))
    vpip("--upgrade", "pip")
    vpip("torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
         "--index-url", "https://download.pytorch.org/whl/cu124")
    vpip("-c", constraints,                 # no bare "triton": torch 2.6.0 brings 3.2.0
         "transformers==4.47.1", "accelerate>=0.25,<2", "datasets>=2.16,<3", "huggingface_hub>=0.24,<1.0",
         "sentencepiece", "protobuf", "einops>=0.7", "scipy", "pyyaml", "tqdm", "pandas", "matplotlib",
         "rouge", "fuzzywuzzy", "python-Levenshtein", "jieba", "pytest")
    check_pins()

    tv, abi, py = subprocess.check_output([PY, "-c",
        "import torch, sys; print(torch.__version__.split('+')[0]); "
        "print(int(torch._C._GLIBCXX_USE_CXX11_ABI)); "
        "print(f'cp{sys.version_info.major}{sys.version_info.minor}')"], env=VENV_ENV, text=True).split()
    tmm = ".".join(tv.split(".")[:2])

    def _fa_loads():
        # The binary must resolve every symbol against THIS torch. Importing the extension is the test;
        # a wheel's file name is not: the 2.8.x "cxx11abiFALSE" wheels for torch 2.6 are built with the
        # new C++ ABI and fail with an undefined c10::Error symbol.
        r = subprocess.run([PY, "-c", "import torch, flash_attn_2_cuda, flash_attn; print(flash_attn.__version__)"],
                           env=VENV_ENV, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout + r.stderr).strip().splitlines()[-1:]

    wheel = None
    for v in ("2.7.4.post1", "2.8.3", "2.8.2"):
        u = (f"https://github.com/Dao-AILab/flash-attention/releases/download/v{v}/"
             f"flash_attn-{v}+cu12torch{tmm}cxx11abi{'TRUE' if abi == '1' else 'FALSE'}-{py}-{py}-linux_x86_64.whl")
        try:
            vpip("--no-deps", "-c", constraints, u)    # --no-deps: its only deps (torch, einops) are in
        except subprocess.CalledProcessError:
            print("   no such wheel")
            continue
        ok, why = _fa_loads()
        if ok:
            wheel = u
            break
        print("   installs but does not load against torch", tv, "->", *why)
        sh(PY, "-m", "pip", "uninstall", "-y", "-q", "flash-attn")
    if wheel:
        print("flash-attn wheel:", wheel.rsplit("/", 1)[1])
    elif FLASH_ATTN_FROM_SOURCE:
        vpip("-c", constraints, "ninja", "packaging", "wheel", "psutil", "setuptools")
        # FORCE_BUILD: otherwise setup.py fetches the same prebuilt wheel that just failed to load
        sh(PY, "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-c", constraints,
           "flash-attn==2.7.4.post1", env=dict(VENV_ENV, MAX_JOBS="8", FLASH_ATTENTION_FORCE_BUILD="TRUE"))
        ok, why = _fa_loads()
        if not ok:
            raise RuntimeError(f"flash-attn built from source does not load: {why}")
    else:
        raise RuntimeError(f"No prebuilt flash-attn wheel loads against torch {tv} / {py} / cxx11abi={abi}. "
                           "Set FLASH_ATTN_FROM_SOURCE = True (about 1-2 h) and rerun this cell.")
print("venv:", check_pins())

check = r"""
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| cxx11abi", torch._C._GLIBCXX_USE_CXX11_ABI)
assert torch.__version__.startswith("2.6."), "torch must be 2.6.x: the eviction does not compile on 2.7.1"
assert torch.cuda.is_available(), "CUDA is not available inside the venv"
import triton, transformers, flash_attn, datasets
print("triton", triton.__version__, "| transformers", transformers.__version__,
      "| flash_attn", flash_attn.__version__, "| datasets", datasets.__version__)
from flash_attn import flash_attn_func
x = torch.randn(1, 256, 8, 128, device="cuda", dtype=torch.float16)
flash_attn_func(x, x, x, causal=True); torch.cuda.synchronize()
f = torch.compile(lambda t: (t * 2).sin().sum(-1))     # the eviction is compiled-or-raise
f(torch.randn(64, 64, device="cuda")); torch.cuda.synchronize()
from utils.cache_factory import assert_transformers_version_supported
assert_transformers_version_supported()
print("environment ok")
"""
r = subprocess.run([PY, "-c", check], cwd=REPO, env=VENV_ENV, capture_output=True, text=True)
print(r.stdout, r.stderr[-4000:] if r.returncode else "")
if r.returncode != 0:
    raise RuntimeError("environment check failed inside the venv; see the output above")
MARK.write_text(STACK_ID)

# The scoring cells (section 8) run in this kernel: they need the LongBench metric packages here too.
missing = [m for m in ("rouge", "fuzzywuzzy", "Levenshtein") if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([KERNEL_PY, "-m", "pip", "install", "-q", "rouge", "fuzzywuzzy", "python-Levenshtein"],
                   check=True)
print("kernel:", sys.version.split()[0], "| jobs:", PY)

## 3. Model and data

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
HF_TOKEN = _secret("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Add a Colab secret HF_TOKEN with access to meta-llama/Llama-3.1-8B-Instruct.")
assert "instruct" in Path(MODEL_DIR).name.lower(), "MODEL_DIR's name must contain 'instruct'"
snapshot_download(MODEL_ID, local_dir=MODEL_DIR, token=HF_TOKEN,
                  allow_patterns=["*.json", "*.safetensors", "tokenizer*"])

LB_DIR = REPO / "data" / "longbench"
LB_DIR.mkdir(parents=True, exist_ok=True)
need = [t for t in TASKS if not (LB_DIR / f"{t}.jsonl").exists()]
if need:
    zp = hf_hub_download("THUDM/LongBench", "data.zip", repo_type="dataset", token=HF_TOKEN)
    with zipfile.ZipFile(zp) as z:                      # members are data/<task>.jsonl
        names = z.namelist()
        for t in need:
            member = next(n for n in names if n == f"{t}.jsonl" or n.endswith(f"/{t}.jsonl"))
            with z.open(member) as src, open(LB_DIR / f"{t}.jsonl", "wb") as dst:
                shutil.copyfileobj(src, dst)
for t in TASKS:
    ids = [json.loads(l)["_id"] for l in open(LB_DIR / f"{t}.jsonl")]
    assert len(ids) == len(set(ids)), f"{t}: duplicate _id, paired comparisons would be wrong"
    print(f"{t:12s} {len(ids)} examples, unique _id")

ENV = dict(VENV_ENV, PYTHONHASHSEED="0", CUBLAS_WORKSPACE_CONFIG=":4096:8",
           TOKENIZERS_PARALLELISM="false", LONGBENCH_LOCAL_DIR=str(LB_DIR),
           HF_TOKEN=HF_TOKEN, PYTHONUNBUFFERED="1")

## 4. Readiness check and batch plan (refuses to start if anything is missing)

1. The gate is wired: `fused_gate` has non-test callers.
2. Every config is at the shipped operating point. This run overrides only the budget, at invocation, and every sidecar
   is checked for 0.10.
3. The knobs exist: the completion knob on `CacheConfig`, and batching and recording on `LongBenchConfig`.
4. The collector has `--completion-rungs`.
5. Preflight tests pass on this GPU:
   * the completion-ablation suite;
   * the batched recorder (B = 2 against B = 1 and against the collector's path);
   * the batched runner (batched = one at a time, truncation, EOS cut, OOM halving);
   * the gate/fill suites and the sm80 compile tests.
6. **The batch plan**, computed with the runner's own prompt encoder, per task:
   * how many of the first examples are too short and are skipped;
   * how much the 50 used are truncated;
   * the batches.

   The plan refuses to start if any task has fewer than `NUM_SAMPLES` examples long enough.

In [ ]:
probe = r"""
import json, dataclasses, subprocess, pathlib
from utils.config import CacheConfig, LongBenchConfig
fields = {f.name for f in dataclasses.fields(CacheConfig)}
lb_fields = {f.name for f in dataclasses.fields(LongBenchConfig)}
callers = subprocess.run("grep -rn 'fused_gate' --include='*.py' . | grep -v '^./tests/'",
                         shell=True, capture_output=True, text=True).stdout.splitlines()
src = pathlib.Path("modules/evaluation/observation_collector.py").read_text()
knobs = ("quant_completion", "quant_completion_scale", "quant_completion_sigma",
         "quant_completion_seed", "quant_completion_scores")
lb_knobs = ("prompt_tokens", "batch_size", "batch_tokens", "completion_rungs", "completion_rungs_stride")
print(json.dumps({"knobs": {k: k in fields for k in knobs}, "lb_knobs": {k: k in lb_fields for k in lb_knobs},
                  "recorder": pathlib.Path("modules/evaluation/completion_recorder.py").exists(),
                  "gate_callers": [c for c in callers if "def fused_gate" not in c],
                  "exp1b_rungs": "--completion-rungs" in src}))
"""
info = json.loads(subprocess.check_output([PY, "-c", probe], cwd=REPO, env=ENV, text=True))
op = subprocess.run([PY, "scripts/check_operating_point.py"], cwd=REPO, env=ENV,
                    capture_output=True, text=True)
READY = {"gate_wired": bool(info["gate_callers"]), "operating_point": op.returncode == 0,
         "knob": all(info["knobs"].values()), "batching": all(info["lb_knobs"].values()),
         "recorder": info["recorder"], "exp1b": info["exp1b_rungs"]}
print(f"commit {SHA[:10]}: " + ", ".join(f"{k}={v}" for k, v in READY.items()))
if not all(READY.values()):
    print(op.stdout[-2000:])
    raise RuntimeError("This commit cannot run the experiment: " +
                       ", ".join(k for k, v in READY.items() if not v) + ". Nothing was run.")
if RUN_PREFLIGHT_TESTS:
    t = subprocess.run([PY, "-m", "pytest", "-q", "-x", "-p", "no:cacheprovider",
                        "tests/test_completion_ablation.py", "tests/test_completion_recorder.py",
                        "tests/test_longbench_batched.py", "tests/test_gate_fill_calibration.py",
                        "tests/test_gate_union_is_per_head.py", "tests/test_kernel_compiles.py"],
                       cwd=REPO, env=dict(ENV, ATEN_CPU_CAPABILITY="default"),  # scalar CPU codegen for the
                       capture_output=True, text=True)                           # CPU-side cache tests only
    print(t.stdout[-1200:])
    if t.returncode != 0:
        raise RuntimeError("Preflight tests failed on this GPU. See the output above.")

# The batch plan, with the runner's own encoder (the same code path the arms run).
plan_py = r"""
import json, sys, types
from transformers import AutoTokenizer
from data.longbench_loader import load_longbench_dataset
from modules.evaluation.longbench_runner import LongBenchRunner
from modules.evaluation.longgenbench_batching import plan_batches
a = json.loads(sys.argv[1])
r = object.__new__(LongBenchRunner)
r.config = types.SimpleNamespace(model=types.SimpleNamespace(name=a["model"]))
r.lb = types.SimpleNamespace(max_length=None)
r.tokenizer = AutoTokenizer.from_pretrained(a["model"])
P = json.load(open("data/longbench_configs/dataset2prompt.json"))
out = {}
for t in a["tasks"]:
    L, rows, skipped, orig = a["prompt_tokens"][t], [], [], []
    for i, ex in enumerate(load_longbench_dataset(t)):
        if len(rows) >= a["n"]:
            break
        n = len(r._encode_example(ex, P[t], t, thudm_truncate=False).input_ids[0])
        (rows if n >= L else skipped).append(ex.get("_id", str(i)))
        if n >= L:
            orig.append(n)
    cap = max(1, min(a["batch_size"], a["batch_tokens"] // L))
    out[t] = {"used": len(rows), "skipped": len(skipped), "scanned": len(rows) + len(skipped),
              "median_full": sorted(orig)[len(orig) // 2] if orig else None,
              "kept_share": (len(orig) * L) / sum(orig) if orig else None,
              "batches": [len(b) for b in plan_batches({k: L for k in range(len(rows))}, cap)]}
print("PLAN" + json.dumps(out))
"""
arg = json.dumps(dict(model=MODEL_DIR, tasks=TASKS, prompt_tokens=PROMPT_TOKENS, n=NUM_SAMPLES,
                      batch_size=BATCH_SIZE, batch_tokens=BATCH_TOKENS))
pr = subprocess.run([PY, "-c", plan_py, arg], cwd=REPO, env=ENV, capture_output=True, text=True)
line = [l for l in pr.stdout.splitlines() if l.startswith("PLAN")]
if pr.returncode or not line:
    raise RuntimeError("batch planning failed:\n" + pr.stderr[-3000:])
PLAN = json.loads(line[0][4:])
print(f"\nbatch plan (BATCH_SIZE {BATCH_SIZE}, BATCH_TOKENS {BATCH_TOKENS}):")
print(f"{'task':12s}{'prompt':>8s}{'used':>6s}{'skipped':>9s}{'median full':>13s}{'text kept':>11s}  batches")
for t, p in PLAN.items():
    print(f"{t:12s}{PROMPT_TOKENS[t]:>8d}{p['used']:>6d}{p['skipped']:>9d}{p['median_full'] or 0:>13d}"
          f"{p['kept_share'] or 0:>10.0%}   {p['batches']}")
short = [t for t, p in PLAN.items() if p["used"] < NUM_SAMPLES]
if short:
    raise RuntimeError(f"{short}: fewer than {NUM_SAMPLES} examples are at least PROMPT_TOKENS long. "
                       "Lower PROMPT_TOKENS for them, or NUM_SAMPLES.")
print("ready")

## 5. Arms and the batched job runner

In [ ]:
LOGS, RUNS = ROOT / "logs", Path("/content/runs")

class Blocked(Exception):
    pass

def run(cmd, log_name):
    cmd = [str(c) for c in cmd]
    log = LOGS / f"{log_name}.log"
    print("$", " ".join(cmd))
    t0, tail = time.time(), []
    keep = ("examples done", "read gate", "OOM", "out of memory", "Error", "error:", "NOT-GATED", "Traceback",
            "score=", "matched-state records")
    with open(log, "w") as fh:
        p = subprocess.Popen(cmd, cwd=REPO, env=ENV, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            fh.write(line)
            tail = (tail + [line])[-60:]
            if any(k in line for k in keep):
                print("   ", line.rstrip()[:200])
        rc = p.wait()
    if rc != 0:
        print("".join(tail))
        raise RuntimeError(f"{log_name} failed (rc={rc}). Full log: {log}")
    print(f"    finished in {(time.time() - t0) / 60:.1f} min")

NATIVE = {"mode": "native", "scale": 1.0, "sigma": 0.0, "seed": 0, "scores": "output"}
def C(**kw):
    return {"mode": "calibrated", "scale": 1.0, "sigma": 0.0, "seed": 0, "scores": "output", **kw}

ARMS = {}
def arm(name, stage, exp, gate=GATE, comp=NATIVE, sigma_mult=None, floor=False, record=False):
    ARMS[name] = dict(stage=stage, exp=exp, gate=gate, comp=comp, sigma_mult=sigma_mult, floor=floor, record=record)

# Run order = priority: a run cut short still has the comparisons that matter most.
arm("cal",              "core",     "control")
arm("off",              "core",     "2", comp=C(mode="off"))
arm("ceiling_g1.0",     "extended", "2", gate=1.0)                       # nothing unread
arm("stress_g0.10_cal", "extended", "2", gate=STRESS_GATE)
arm("stress_g0.10_off", "extended", "2", gate=STRESS_GATE, comp=C(mode="off"))
arm("rec_g0.25",        "collect",  "1", record=True)                    # cal, recording every variant
arm("rec_g0.10",        "collect",  "1", gate=STRESS_GATE, record=True)  # stress_g0.10_cal, recording
arm("uncal",            "core",     "3", comp=C(mode="uncalibrated"))
arm("scale2",           "core",     "4A", comp=C(scale=2.0))
arm("noise1x_s0",       "core",     "4B", comp=C(seed=0), sigma_mult=1)
arm("cal_repeat",       "core",     "floor", floor=True)                 # run-to-run noise
arm("cal_ep",           "core",     "floor", comp=C(), floor=True)       # the epilogue at identity
for c in (0.5, 0.8, 1.25, 4.0):
    arm(f"scale{c:g}", "extended", "4A", comp=C(scale=c))
for s in NOISE_SEEDS:                     # seed-major: every level gets seed 0 before any gets seed 1
    for m in NOISE_MULTS:
        if f"noise{m}x_s{s}" not in ARMS:
            arm(f"noise{m}x_s{s}", "extended" if s == 0 or m == 1 else "followup", "4B",
                comp=C(seed=s), sigma_mult=m)
for c in (0.5, 2.0):
    arm(f"coupled_scale{c:g}", "extended", "4C", comp=C(scale=c, scores="coupled"))
TWIN = {"rec_g0.25": "cal", "rec_g0.10": "stress_g0.10_cal"}   # recording must change nothing

SIGMA_FILE = ROOT / "exp1" / "sigma_hat.json"

def selected():
    return [n for n, a in ARMS.items() if a["stage"] in STAGES[RUN]]

def n_for(a):
    return NUM_SAMPLES_FLOOR if a["floor"] else NUM_SAMPLES

def resolved_comp(a):
    c = dict(a["comp"])
    if a["sigma_mult"]:
        if not SIGMA_FILE.exists():
            raise Blocked("noise arms need sigma_hat_U from Experiment 1. Run section 6 first")
        c["sigma"] = a["sigma_mult"] * json.loads(SIGMA_FILE.read_text())["sigma_hat_U"]
    return c

def overrides(a, comp):
    ov = [f"model.name={MODEL_DIR}", f"cache.cache_budget={CACHE_BUDGET}", f"cache.quant_gate_ratio={a['gate']}",
          f"longbench.batch_size={BATCH_SIZE}", f"longbench.batch_tokens={BATCH_TOKENS}",
          "longbench.aggressive_cache_clear=true",
          *[f"longbench.prompt_tokens.{t}={PROMPT_TOKENS[t]}" for t in TASKS]]
    if a["record"]:
        ov += ["longbench.completion_rungs=true", f"longbench.completion_rungs_stride={RECORD_STRIDE}"]
    if comp["mode"] != "native":
        ov += [f"cache.quant_completion={comp['mode']}", f"cache.quant_completion_scale={comp['scale']}",
               f"cache.quant_completion_sigma={comp['sigma']}", f"cache.quant_completion_seed={comp['seed']}",
               f"cache.quant_completion_scores={comp['scores']}"]
    return ov

def _same(g, v):
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return g is not None and abs(float(g) - float(v)) <= 1e-9 * max(1.0, abs(float(v)))
    return g == v

def verify(meta, a, comp, n, task):
    """Everything a sidecar must say for its number to belong to this arm."""
    errs = []
    rg = meta.get("read_gate") or {}
    if rg.get("verdict") != "gated":
        errs.append(f"read_gate verdict {rg.get('verdict')!r}, need 'gated'")
    rf = rg.get("read_fraction")
    if rf is None or not (a["gate"] - 1e-6 <= rf <= min(1.0, a["gate"] + 0.15) + 1e-6):
        errs.append(f"realised read fraction {rf} for requested ratio {a['gate']}")
    for k, v in OPERATING_POINT.items():
        if not _same(meta.get(k), v):
            errs.append(f"{k} = {meta.get(k)!r}, expected {v!r}")
    got = meta.get("quant_completion")
    if not isinstance(got, dict):
        errs.append("no quant_completion record in the sidecar")
    else:
        for k, v in comp.items():
            if not _same(got.get(k), v):
                errs.append(f"quant_completion.{k} = {got.get(k)!r}, expected {v!r}")
    applied, gated = rg.get("completion_applied"), rg.get("gated")
    want = 0 if comp["mode"] == "native" else gated
    if applied is None or applied != want or (comp["mode"] != "native" and not gated):
        errs.append(f"completion_applied = {applied}, gated = {gated}: the epilogue must run on every "
                    "gated layer of a variant arm and on none of a native one")
    if meta.get("num_samples_requested") != n:
        errs.append(f"num_samples_requested = {meta.get('num_samples_requested')!r}, expected {n!r}")
    if meta.get("num_examples") != n:
        errs.append(f"num_examples = {meta.get('num_examples')!r}: fewer than {n} examples were long enough")
    b = meta.get("batching") or {}
    if b.get("prompt_tokens") != PROMPT_TOKENS[task]:
        errs.append(f"prompt_tokens = {b.get('prompt_tokens')!r}, expected {PROMPT_TOKENS[task]}")
    cr = b.get("completion_rungs")
    if a["record"] and not (cr and cr.get("stride") == RECORD_STRIDE and cr.get("files")):
        errs.append(f"no matched-state records (completion_rungs = {cr!r})")
    if not a["record"] and cr:
        errs.append("the recorder ran on an arm that does not record")
    return errs

def _refuse(files, tag, errs):
    dst = ROOT / "refused" / tag
    dst.mkdir(parents=True, exist_ok=True)
    for f in files:
        if Path(f).exists():
            shutil.move(str(f), dst / Path(f).name)
    raise RuntimeError(f"REFUSED {tag}:\n  " + "\n  ".join(errs) + f"\n  files moved to {dst}")

def lb_arm(name):
    """Every pending task of one arm in ONE process (one model load), batched, then each task's
    sidecar verified and copied to Drive. A task that finished before a crash is kept."""
    a = ARMS[name]
    comp, n = resolved_comp(a), n_for(a)
    dst = ROOT / "longbench" / name
    dst.mkdir(parents=True, exist_ok=True)
    todo = []
    for t in TASKS:
        meta_f, pred_f = dst / f"{t}.meta.json", dst / f"{t}.jsonl"
        if meta_f.exists() and pred_f.exists():     # the sidecar is written last: it is the done-marker
            errs = verify(json.loads(meta_f.read_text()), a, comp, n, t)
            if errs:
                raise RuntimeError(f"STALE {name}/{t} on Drive does not match this arm:\n  "
                                   + "\n  ".join(errs) + "\n  Move it away to rerun.")
        else:
            todo.append(t)
    if not todo:
        return "done"
    local = RUNS / "longbench" / name
    shutil.rmtree(local, ignore_errors=True)    # the runner's own resume trusts a partial jsonl
    local.mkdir(parents=True)
    cmd = [PY, "main.py", "--config", "configs/longbench_ours_flash_attn.yaml", "--override",
           f"longbench.datasets=[{','.join(todo)}]", f"longbench.output_dir={local}",
           f"longbench.num_samples={n}", "longbench.resume=false",
           f"longbench.skip_oom={str(SKIP_OOM).lower()}", *overrides(a, comp)]
    failure = None
    try:
        run(cmd, f"lb_{name}")
    except RuntimeError as e:
        failure = e
    for t in todo:
        mf, pf = local / f"{t}.meta.json", local / f"{t}.jsonl"
        if not (mf.exists() and pf.exists()):
            continue
        errs = verify(json.loads(mf.read_text()), a, comp, n, t)
        if errs:
            _refuse([pf, mf], f"{name}_{t}", errs)
        if a["record"]:                              # the matched-state records, before the done-marker
            shutil.copytree(local / "rungs" / t, dst / "rungs" / t, dirs_exist_ok=True)
        shutil.copy(pf, dst / f"{t}.jsonl")
        shutil.copy(mf, dst / f"{t}.meta.json")
    if failure is not None:
        raise failure
    return "ran"

def status_table():
    names = selected()
    eps, remaining_s, loads = {}, 0.0, 0
    for t in TASKS:   # measured examples/second per task, from any finished native arm
        for nm in ("cal", "ceiling_g1.0", "stress_g0.10_cal"):
            f = ROOT / "longbench" / nm / f"{t}.meta.json"
            if f.exists():
                eps[t] = json.loads(f.read_text()).get("examples_per_second")
                break
    print(f"\n{'arm':20s}{'stage':>10s}" + "".join(f"{t:>13s}" for t in TASKS))
    for nm in names:
        cells = []
        for t in TASKS:
            done = (ROOT / "longbench" / nm / f"{t}.meta.json").exists()
            cells.append("done" if done else "-")
            if not done and eps.get(t):
                a = ARMS[nm]
                remaining_s += n_for(a) / eps[t] * (1.0 if a["comp"]["mode"] == "native" else 1.2) \
                    * (1.5 if a["record"] else 1.0)
        print(f"{nm:20s}{ARMS[nm]['stage']:>10s}" + "".join(f"{c:>13s}" for c in cells))
        loads += "-" in cells
    if eps:
        remaining_s += 90 * loads                  # one model load + compile warm-up per pending arm
        print(f"\nprojected remaining for RUN = {RUN!r}: {remaining_s / 3600:.1f} h (measured ex/s of finished "
              "native arms; +20% for epilogue arms, +50% for recorder arms; ~90 s load per arm)")

def run_grid():
    failed = []
    for nm in selected():
        try:
            if lb_arm(nm) == "ran":
                status_table()                      # progress, and the re-projected remaining time
        except Blocked as e:
            print(f"BLOCKED {nm}: {e}")
        except RuntimeError as e:                   # keep going: the other arms do not depend on this one
            print(f"FAILED {nm}: {str(e)[:2000]}")
            failed.append(nm)
    status_table()
    if failed:
        raise RuntimeError(f"{len(failed)} arm(s) failed: {failed}. Their logs are in {LOGS}; run this cell "
                           "again to retry them (finished arms are skipped).")

## 6. Experiment 1 (the collector): every variant on the same decode states, at ρ = 0.25 and ρ = 0.10

This is one collector run per task and read ratio on the shipped method, with `--completion-rungs`. At every gated
(step, layer, query head), the collector holds the dequantized int2 tier and the full-KV shadow, and computes **on the same
state**:
* the exact int2 mass of every window over `Z_A`, `m*(u)`;
* the calibrated and uncalibrated masses;
* the output of every variant: `off`, `uncal`, `cal`, five scales and four noise levels, the **mass oracle** (`m*` with
  the stored centroid), the **value oracle**, and the **full oracle** (attention over every held window).

Output errors are measured against `o_held`, which removes eviction and int2 error and isolates the completion term.
They are normalised by the full-KV output.

| Statistic | Meaning |
|---|---|
| `unread` | share of held attention on unread int2 windows: what the completion stands in for |
| `E_w`, `E_unw` | `|log m̃_cal − log m*|` over unread windows, mass-weighted and unweighted |
| `bias_w` | mean signed log error on the unread windows (the selection bias) |
| `tot_err` | error on the total unread mass. `c_hat = exp(median)` is marked on the scale plot |
| `sd_U` | residual spread on the unread windows. Its median at ρ = 0.25 is `σ̂_U`, the unit of the noise arms |
| rungs | rel. L2 output error of each variant vs `o_held` |

Sanity gates that refuse the data:
* `full_oracle` must reproduce `o_held` (≤ 1e-3);
* `cal` must reproduce the kernel's real output: median `cal_vs_actual` ≤ `EXP1_CAL_VS_KERNEL_MAX`.

Statistics are taken per query head first, then aggregated over heads. **Caveat:** each prompt is the first
`EXP1_PREFILL[task]` tokens of a document plus a greedy continuation, not the templated task prompt. The recorder arms in
section 7 (`rec_g0.25`, `rec_g0.10`) remove that caveat: they record the same statistics inside the real batched runs,
on all 50 templated prompts per task and every 8th step of their long generations. Section 8 reads both.

`sigma_hat.json` is written once, from ρ = 0.25, and never rewritten: the noise arms run at the value it holds.

In [ ]:
import numpy as np, pandas as pd

FLOOR = 1e-6

def iter_zip(path, keys):
    with zipfile.ZipFile(path) as zf:
        meta = json.loads(zf.read("meta.json"))
        for s in meta["samples"]:
            with np.load(io.BytesIO(zf.read(s["file"])), allow_pickle=False) as z:
                yield {k: z[k] for k in keys}

def _head_step(tier_tl, go_tl, lms, err, shift, cerr, cva, rep):
    """One (step, layer): a row of statistics per query head (NaN where undefined)."""
    q = (tier_tl == 1)[None, :]
    op = np.repeat(go_tl, rep, axis=0)
    R, U = q & op, q & ~op
    lms = lms.astype(np.float64); e = err.astype(np.float64)
    fin = np.isfinite(lms) & np.isfinite(e)
    m = np.where(fin, np.exp(np.where(fin, lms, -np.inf)), 0.0)
    Uw, Uf = U & fin, U & fin & (m >= FLOOR)
    nU = Uf.sum(1)
    with np.errstate(invalid="ignore", divide="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        wU = np.where(Uw, m, 0.0); sw = wU.sum(1)
        e_unc = e + shift[:, None]
        out = {
            "unread": sw / (1.0 + sw),
            "E_w": (wU * np.abs(np.where(Uw, e, 0))).sum(1) / sw,
            "E_unw": np.where(nU > 0, (np.abs(np.where(Uf, e, 0))).sum(1) / nU, np.nan),
            "bias_w": (wU * np.where(Uw, e, 0)).sum(1) / sw,
            "bias_unw": np.where(nU > 0, np.where(Uf, e, 0).sum(1) / nU, np.nan),
            "tot_err": np.log((wU * np.exp(np.where(Uw, e, 0))).sum(1)) - np.log(sw),
            "sd_U": np.where(nU >= 4, np.nanstd(np.where(Uf, e, np.nan), axis=1, ddof=1), np.nan),
            "E_w_uncal": (wU * np.abs(np.where(Uw, e_unc, 0))).sum(1) / sw,
            "uncal_shift": shift.astype(np.float64),
            "cal_vs_actual": cva.astype(np.float64),
        }
    valid = (R.sum(1) > 0) & (Uw.sum(1) > 0)
    for k in out:
        out[k] = np.where(valid & np.isfinite(out[k]), out[k], np.nan)
    rung = np.where(valid[:, None], cerr[..., 0].astype(np.float64), np.nan)     # vs o_held
    rung_full = np.where(valid[:, None], cerr[..., 1].astype(np.float64), np.nan)
    return out, rung, rung_full

EXP1_KEYS = ("tier", "gate_open", "gate_fired", "comp_log_mstar", "comp_logerr_cal",
             "comp_uncal_shift", "comp_err", "comp_cal_vs_actual")

def exp1_analyse(zp):
    with zipfile.ZipFile(zp) as zf:
        meta = json.loads(zf.read("meta.json"))
    rungs = meta["completion_rungs"]
    stats, rg, rgf, ids = {}, [], [], []
    for s in iter_zip(zp, EXP1_KEYS):
        T, L, H, _ = s["comp_log_mstar"].shape
        rep = H // s["gate_open"].shape[2]
        for t in range(1, T):
            for l in np.flatnonzero(s["gate_fired"][t]):
                o, r, rf = _head_step(s["tier"][t, l], s["gate_open"][t, l], s["comp_log_mstar"][t, l],
                                      s["comp_logerr_cal"][t, l], s["comp_uncal_shift"][t, l],
                                      s["comp_err"][t, l], s["comp_cal_vs_actual"][t, l], rep)
                for k, v in o.items():
                    stats.setdefault(k, []).append(v.astype(np.float32))
                rg.append(r.astype(np.float32)); rgf.append(rf.astype(np.float32))
                ids.append(np.stack([np.full(H, l), np.arange(H)], 1))
    ids = np.concatenate(ids)
    df = pd.DataFrame({k: np.concatenate(v) for k, v in stats.items()})
    df["layer"], df["head"] = ids[:, 0], ids[:, 1]
    R = pd.DataFrame(np.concatenate(rg), columns=rungs)
    Rf = pd.DataFrame(np.concatenate(rgf), columns=rungs)
    R["layer"] = Rf["layer"] = ids[:, 0]
    R["head"] = Rf["head"] = ids[:, 1]
    return meta, df, R, Rf

def summarise(df):
    ph = df.groupby(["layer", "head"]).median()
    worst = {k: (ph[k].abs().max() if k in ("bias_w", "bias_unw", "tot_err") else ph[k].max()) for k in ph}
    return pd.DataFrame({"median head": ph.median(), "p90 head": ph.quantile(0.9), "worst head": pd.Series(worst)})

def exp1_job(task, gate):
    out_dir = ROOT / "exp1" / f"g{gate:.2f}"
    res = out_dir / f"{task}_results.json"
    if res.exists():
        return json.loads(res.read_text())
    out_dir.mkdir(parents=True, exist_ok=True)
    zp = RUNS / "exp1" / f"g{gate:.2f}" / f"{task}_p{EXP1_PREFILL[task]}_n{EXP1_GEN}_k{EXP1_PROMPTS}.zip"
    zp.parent.mkdir(parents=True, exist_ok=True)
    if not zp.exists():
        run(["bash", "scripts/collect_observations.sh", "--model-path", MODEL_DIR, "--dataset", task,
             "--window-size", OPERATING_POINT["window_size"], "--cache-budget", CACHE_BUDGET,
             "--gate-ratio", gate, "--quant-ratio", OPERATING_POINT["quant_ratio"],
             "--quant-budget-mode", OPERATING_POINT["quant_budget_mode"],
             "--prefill", EXP1_PREFILL[task], "--gen", EXP1_GEN, "--num-samples", EXP1_PROMPTS,
             "--completion-rungs", "--out", zp], f"exp1_g{gate:.2f}_{task}")
    meta, df, R, Rf = exp1_analyse(zp)
    errs = []
    rg = meta.get("read_gate") or {}
    if rg.get("verdict") != "gated":
        errs.append(f"collector read_gate verdict {rg.get('verdict')!r}")
    rf = rg.get("read_fraction")
    if rf is None or not (gate - 1e-6 <= rf <= min(1.0, gate + 0.15) + 1e-6):
        errs.append(f"collector realised read fraction {rf} for requested ratio {gate}")
    fo = float(np.nanmax(R["full_oracle"])) if len(R) else float("nan")
    if not fo <= 1e-3:
        errs.append(f"full_oracle differs from the collector's o_held by up to {fo:.2e}: the rung arithmetic is wrong")
    cva = float(np.nanmedian(df["cal_vs_actual"]))
    if not cva <= EXP1_CAL_VS_KERNEL_MAX:
        errs.append(f"the recomputed calibrated output differs from the kernel's real output by {cva:.2e} "
                    f"(median) > {EXP1_CAL_VS_KERNEL_MAX}: the rungs would describe a different computation "
                    "from the one that ran")
    if errs:
        _refuse([zp], f"exp1_g{gate:.2f}_{task}", errs)
    ph = R.groupby(["layer", "head"]).median()
    rung_tab = pd.DataFrame({"median head": ph.median(), "p90 head": ph.quantile(0.9),
                             "vs full-KV, median head": Rf.groupby(["layer", "head"]).median().median()}
                            ).drop(index=["layer", "head"], errors="ignore")
    out = {"task": task, "gate": gate, "source": "collector", "head_steps": int(len(df)),
           "summary": summarise(df).to_dict(), "rungs": rung_tab.to_dict(),
           "sigma_hat_U": float(np.nanmedian(df["sd_U"])), "tot_err_median": float(np.nanmedian(df["tot_err"])),
           "uncal_shift_median": float(np.nanmedian(df["uncal_shift"])),
           "cal_vs_actual_median": cva,
           "cal_vs_actual_p99": float(np.nanpercentile(df["cal_vs_actual"].dropna(), 99))
           if df["cal_vs_actual"].notna().any() else None,
           "full_oracle_max": fo, "read_fraction": rf, "sha": SHA}
    df.groupby(["layer", "head"]).median().to_csv(out_dir / f"{task}_per_head.csv")
    ph.to_csv(out_dir / f"{task}_rungs_per_head.csv")
    res.write_text(json.dumps(out, indent=1, default=float))
    return out

def show_exp1(E, label):
    for t, r in E.items():
        print(f"\n=== {label} {t}: {r['head_steps']} head-steps   cal vs kernel output: median "
              f"{r['cal_vs_actual_median']:.2e}   full oracle vs o_held: max {r['full_oracle_max']:.1e}")
        print(pd.DataFrame(r["summary"]).round(4).to_string())
        print(pd.DataFrame(r["rungs"]).round(5).to_string())
    for t, r in E.items():
        m = r["rungs"]["median head"]
        print(f"{label} {t:12s} completion removes {1 - m['cal'] / m['off']:.1%} of the read-gate output error "
              f"(off {m['off']:.4f} -> cal {m['cal']:.4f}; mass oracle {m['mass_oracle']:.4f})")

E1 = {}
for g in EXP1_GATES:
    E1[g] = {t: exp1_job(t, g) for t in TASKS}
    show_exp1(E1[g], f"rho {g:.2f}")
c = E1[GATE]
sig = float(np.median([r["sigma_hat_U"] for r in c.values()]))
rec = {"sigma_hat_U": sig, "c_hat": float(np.exp(np.median([r["tot_err_median"] for r in c.values()]))),
       "uncal_factor": float(np.exp(np.median([r["uncal_shift_median"] for r in c.values()]))), "floor": FLOOR,
       "gate": GATE, "sha": SHA,
       "per_task": {t: {k: r[k] for k in ("sigma_hat_U", "tot_err_median", "uncal_shift_median")} for t, r in c.items()}}
if SIGMA_FILE.exists():                # never rewritten: every noise arm on Drive ran at exactly this value
    old = json.loads(SIGMA_FILE.read_text())
    if abs(old["sigma_hat_U"] - sig) > 1e-9:
        raise RuntimeError(f"sigma_hat_U changed ({old['sigma_hat_U']} -> {sig}); noise arms already on Drive were "
                           "run at the old value. Keep the old Experiment 1 results or start a new DRIVE_ROOT.")
else:
    SIGMA_FILE.write_text(json.dumps(rec, indent=2))
print(f"\nsigma_hat_U = {sig:.3f} nats (noise-arm unit)   c_hat = {rec['c_hat']:.3f}   "
      f"uncal factor = {rec['uncal_factor']:.3f}")

## 7. The LongBench arms, batched: every arm for the current `RUN`

There is one process per arm (one model load), and its tasks run in order:
* every task's 50 rows run in batches of up to `BATCH_SIZE`, capped at `BATCH_TOKENS` tokens per batch;
* a batch that runs out of memory is halved, and the event is recorded;
* each task's sidecar is verified and copied to Drive as soon as it exists (the recorder arms copy their `rungs/` first).

Finished arms are skipped. An arm that fails is reported, and the grid moves on to the next one. The cell raises at the
end if anything failed, and running it again retries only those arms. The status table re-projects the remaining time
after every arm from measured examples per second.

In [ ]:
status_table()
run_grid()

## 8. Scoring, paired comparisons, matched states, figures

**Scoring.**
* Per-example scores use the repo's own LongBench metric functions (ROUGE-L for all three tasks).
* Task scores follow THUDM: a null (OOM) prediction scores 0.
* Every sidecar is re-verified, and a file that fails is not scored.

**Paired comparisons.** Every comparison is paired with `cal`, on the examples both arms scored, matched by `_id`. Each
gets a 95% bootstrap CI: tasks are resampled independently and averaged, and every comparison has its own fixed seed, so
a number does not move when other arms are added. Alongside each:
* the minimum detectable effect (80% power, two-sided 5%) at this N;
* the share of examples whose prediction text changed at all.

Read the changes against the two floors (`cal_repeat`, `cal_ep`) and against the recorder arms' twins.

**Derived comparisons.**
* **ρ = 1.0:** the share of the read-gate gap that completion closes, (cal − off)/(ρ1.0 − off). It is defined only
  when ρ = 1.0 beats `off` significantly.
* **Stress:** `cal − off` at each ρ, and the difference-in-differences `(cal − off)@0.10 − (cal − off)@0.25` on the
  examples all four arms share, resampled jointly.
* **Noise:** a level's effect is the seed mean `mean_s(arm_s − cal)` per example, with the spread across seeds alongside.

In [ ]:
import zlib, hashlib
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from modules.evaluation.longbench_scoring import METRIC_FN_REGISTRY, _FIRST_LINE_DATASETS
D2M = json.loads((REPO / "data/longbench_configs/dataset2metric.json").read_text())
RES = ROOT / "results"

def example_scores(jsonl, task):
    fn, out, preds = METRIC_FN_REGISTRY[D2M[task]], {}, {}
    for line in open(jsonl, encoding="utf-8"):
        if not line.strip():
            continue
        ex = json.loads(line)
        pred = ex.get("pred")
        preds[ex["_id"]] = pred
        if pred is None:
            out[ex["_id"]] = None
            continue
        if task in _FIRST_LINE_DATASETS:
            pred = pred.lstrip("\n").split("\n")[0]
        out[ex["_id"]] = 100.0 * max((fn(pred, gt, all_classes=ex.get("all_classes") or [])
                                      for gt in ex.get("answers", [])), default=0.0)
    return out, preds

SCORES, PREDS, META, SKIPPED = {}, {}, {}, []
for name, a in ARMS.items():
    for t in TASKS:
        d = ROOT / "longbench" / name
        if not (d / f"{t}.meta.json").exists():
            continue
        try:
            comp = resolved_comp(a)
        except Blocked:
            continue
        meta = json.loads((d / f"{t}.meta.json").read_text())
        errs = verify(meta, a, comp, n_for(a), t)
        if errs:
            SKIPPED.append((name, t, errs))
            continue
        SCORES[(name, t)], PREDS[(name, t)] = example_scores(d / f"{t}.jsonl", t)
        META[(name, t)] = meta
for s in SKIPPED:
    print("NOT SCORED", s)
if not all(("cal", t) in SCORES for t in TASKS):
    raise RuntimeError("cal is not scored on every task yet: run section 7")
IDS = {t: list(SCORES[("cal", t)]) for t in TASKS}               # example order

def _rng(*key):
    return np.random.default_rng(zlib.crc32("|".join(map(str, key)).encode()))

def have(*names):
    return all((n, t) in SCORES for n in names for t in TASKS)

def _common(names, t):
    """Examples, in order, that every arm in `names` scored non-null."""
    Ss = [SCORES[(n, t)] for n in names]
    return [i for i in IDS[t] if all(S.get(i) is not None for S in Ss)]

def _arrays(names, t):
    ids = _common(names, t)
    return ids, {n: np.array([SCORES[(n, t)][i] for i in ids], dtype=float) for n in names}

def dstats(names, f, key):
    """Paired statistic over the examples every arm in `names` shares; f(A) is the per-example delta for one
    task. Per-task means with 95% CIs; the task average with a 95% CI (tasks resampled independently);
    the MDE at 80% power; n; and whether the CI excludes 0. None if any arm lacks a task."""
    if not have(*names):
        return None
    out, dists, means, ses, n = {"per_task": {}}, [], [], [], 0
    for t in TASKS:
        ids, A = _arrays(names, t)
        if len(ids) < 2:
            return None
        d = f(A)
        bd = d[_rng(*key, t).integers(0, len(d), (BOOTSTRAP_ITERS, len(d)))].mean(1)
        lo, hi = np.percentile(bd, [2.5, 97.5])
        out["per_task"][t] = (float(d.mean()), float(lo), float(hi), len(d))
        dists.append(bd); means.append(d.mean()); ses.append(d.std(ddof=1) / math.sqrt(len(d))); n += len(d)
    avg = np.mean(dists, 0)
    out.update(delta=float(np.mean(means)), lo=float(np.percentile(avg, 2.5)), hi=float(np.percentile(avg, 97.5)),
               mde80=float(2.80 * math.sqrt(sum(s * s for s in ses)) / len(TASKS)), n=n)
    out["sig"] = out["lo"] > 0 or out["hi"] < 0
    return out

def delta(x, y="cal"):
    return dstats((x, y), lambda A: A[x] - A[y], ("delta", x, y))

def ratio_stats(names, fnum, fden, key):
    """Ratio of two task-average deltas, numerator and denominator resampled on the same examples."""
    if not have(*names):
        return None
    nums, dens, pn, pdn = [], [], [], []
    for t in TASKS:
        ids, A = _arrays(names, t)
        idx = _rng(*key, t).integers(0, len(ids), (BOOTSTRAP_ITERS, len(ids)))
        a, b = fnum(A), fden(A)
        nums.append(a[idx].mean(1)); dens.append(b[idx].mean(1)); pn.append(a.mean()); pdn.append(b.mean())
    N, D = np.mean(nums, 0), np.mean(dens, 0)
    with np.errstate(divide="ignore", invalid="ignore"):
        R = N / D
    return dict(point=float(np.mean(pn) / np.mean(pdn)) if np.mean(pdn) else float("nan"),
                lo=float(np.percentile(R, 2.5)), hi=float(np.percentile(R, 97.5)), den_lo=float(np.percentile(D, 2.5)))

def task_score(n, t):
    return float(np.mean([v or 0.0 for v in SCORES[(n, t)].values()]))

def changed(x, y="cal"):
    same = [PREDS[(x, t)][i] == PREDS[(y, t)][i] for t in TASKS for i in _common((x, y), t)]
    return 1.0 - float(np.mean(same)) if same else float("nan")

def similarity(x, y="cal"):
    sim = [difflib.SequenceMatcher(None, PREDS[(x, t)][i], PREDS[(y, t)][i]).ratio()
           for t in TASKS for i in _common((x, y), t)]
    return float(np.mean(sim)) if sim else float("nan")

def ci(r, nd=2):
    return "not run" if r is None else f"{r['delta']:+.{nd}f} [{r['lo']:+.{nd}f}, {r['hi']:+.{nd}f}]"

def dtable(rows):
    out = {}
    for lab, r in rows:
        if r is None:
            out[lab] = {"average": "not run"}
            continue
        out[lab] = {**{t: f"{m:+.2f} [{lo:+.2f},{hi:+.2f}]" for t, (m, lo, hi, _) in r["per_task"].items()},
                    "average": ci(r), "MDE80": round(r["mde80"], 2), "n": r["n"],
                    "CI excl. 0": "yes" if r["sig"] else "no"}
    return pd.DataFrame(out).T

# ---- task scores ----
present = [n for n in ARMS if have(n)]
tab = pd.DataFrame({t: {n: task_score(n, t) for n in present} for t in TASKS})
tab["average"] = tab[TASKS].mean(axis=1)
print(f"Task scores (ROUGE-L x 100, THUDM convention; {NUM_SAMPLES} examples per task):")
print(tab.round(2).to_string())
tab.to_csv(RES / "task_scores.csv")

# ---- every arm vs cal ----
D = {n: delta(n) for n in present if n != "cal"}
DELTA = dtable([(n, D[n]) for n in D])
DELTA["changed"] = [f"{changed(n):.0%}" for n in D]
DELTA["similarity"] = [f"{similarity(n):.3f}" for n in D]
DELTA["exp"] = [ARMS[n]["exp"] for n in D]
print("\nPaired Δ vs cal, points [95% CI]; changed / similarity: prediction text vs cal (read against the floors):")
print(DELTA.to_string())
DELTA.to_csv(RES / "paired_deltas.csv")
if D.get("off"):
    print(f"\nPOWER CHECK: at N = {NUM_SAMPLES}/task, a change of the task average smaller than "
          f"{D['off']['mde80']:.2f} points is not reliably detectable end to end. Experiment 1 carries the "
          "fine-grained answer.")

# ---- recording changes nothing ----
for r_, twin in TWIN.items():
    if have(r_, twin):
        x = delta(r_, twin)
        print(f"recorder passivity: {r_} - {twin} = {ci(x)}, predictions changed {changed(r_, twin):.0%} "
              f"(floor cal_repeat: {changed('cal_repeat') if have('cal_repeat') else float('nan'):.0%})")

# ---- rho = 1.0: the gap closed ----
CEIL, STRESS = "ceiling_g1.0", ("stress_g0.10_cal", "stress_g0.10_off")
GAP = ratio_stats((CEIL, "cal", "off"), lambda A: A["cal"] - A["off"], lambda A: A[CEIL] - A["off"], ("gap",))
if GAP:
    print("\nShare of the read-gate gap closed by completion, (cal - off)/(rho1.0 - off): " +
          (f"{GAP['point']:.2f} [{GAP['lo']:.2f}, {GAP['hi']:.2f}]" if GAP["den_lo"] > 0 else
           "undefined: rho = 1.0 does not beat off significantly at this N"))

# ---- the stress test ----
SC, SO = STRESS
B25 = dstats(("cal", "off"), lambda A: A["cal"] - A["off"], ("benefit", GATE))
B10 = dstats((SC, SO), lambda A: A[SC] - A[SO], ("benefit", STRESS_GATE))
DID = dstats((SC, SO, "cal", "off"), lambda A: (A[SC] - A[SO]) - (A["cal"] - A["off"]), ("did",))
ST = dtable([(f"cal - off @ {GATE:.2f}", B25), (f"cal - off @ {STRESS_GATE:.2f}", B10),
             (f"DiD: (cal-off)@{STRESS_GATE:.2f} - (cal-off)@{GATE:.2f}", DID),
             (f"cal@{STRESS_GATE:.2f} - cal@{GATE:.2f}", delta(SC, "cal") if have(SC) else None),
             (f"off@{STRESS_GATE:.2f} - off@{GATE:.2f}", delta(SO, "off") if have(SO, "off") else None)])
print("\nStress test, paired, points [95% CI]:")
print(ST.to_string())
ST.to_csv(RES / "stress_deltas.csv")

# ---- noise: seed means ----
SEEDS = {}
for m in NOISE_MULTS:
    names = [f"noise{m}x_s{s}" for s in NOISE_SEEDS if have(f"noise{m}x_s{s}")]
    r = dstats(("cal", *names), lambda A, names=names: np.mean([A[x] - A["cal"] for x in names], 0),
               ("noise", m, *names)) if names else None
    SEEDS[m] = (r, {x: D.get(x) for x in names})
    if r:
        sp = [p["delta"] for p in SEEDS[m][1].values() if p]
        print(f"noise {m}x sigma_hat_U, mean of {len(names)} seed(s): {ci(r)}  (single seeds "
              + ", ".join(f"{v:+.2f}" for v in sp) + ")")

In [ ]:
# ---------- matched states: the collector (section 6) and the recorder arms at scale ----------
REC_ARM = {GATE: "rec_g0.25", STRESS_GATE: "rec_g0.10"}

def scale_job(gate):
    """Experiment 1 at scale: the recorder arm's records for every task, reduced like the collector's,
    refused unless cal reproduces the kernel and full_oracle reproduces o_held."""
    name = REC_ARM[gate]
    out_f = ROOT / "exp1" / f"scale_g{gate:.2f}.json"
    if not have(name):
        return None
    files = {t: sorted((ROOT / "longbench" / name / "rungs" / t).glob("*.npz")) for t in TASKS}
    sig = hashlib.sha256(json.dumps({t: [f.name for f in fs] for t, fs in files.items()}).encode()).hexdigest()
    if out_f.exists() and json.loads(out_f.read_text()).get("files_sha") == sig:
        return json.loads(out_f.read_text())
    res = {}
    for t, fs in files.items():
        errs_, stats_, rungs, names = [], [], None, None
        for f in fs:
            with np.load(f, allow_pickle=False) as z:
                gen = z["gen_tokens"][z["rows"].astype(int)]
                keep = (z["steps"] <= gen) & z["fired"].any(1)          # the row's own text, a gated step
                errs_.append(z["err"][keep]); stats_.append(z["stats"][keep])
                rungs, names = [str(x) for x in z["rungs"]], [str(x) for x in z["stat_names"]]
        err, st = np.concatenate(errs_), np.concatenate(stats_)          # [R, L, H, K], [R, L, H, S]
        R_, L_, H_ = st.shape[:3]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            ph_err = np.nanmedian(err, 0).reshape(L_ * H_, -1)          # per head, over records
            ph_st = np.nanmedian(st, 0).reshape(L_ * H_, -1)
            S = {k: st[..., i] for i, k in enumerate(names)}
            fo = float(np.nanmax(err[..., rungs.index("full_oracle")]))
            cva = float(np.nanmedian(S["cal_vs_actual"]))
        bad = []
        if not fo <= 1e-3:
            bad.append(f"full_oracle differs from o_held by up to {fo:.2e}")
        if not cva <= EXP1_CAL_VS_KERNEL_MAX:
            bad.append(f"recomputed calibrated output vs the kernel's: median {cva:.2e}")
        if bad:
            raise RuntimeError(f"REFUSED matched-state records of {name}/{t}: " + "; ".join(bad))
        valid = np.isfinite(st[..., names.index("unread")])
        res[t] = {"task": t, "gate": gate, "source": "recorder", "head_steps": int(valid.sum()),
                  "records": int(R_),
                  "summary": {"median head": dict(zip(names, np.nanmedian(ph_st, 0).tolist())),
                              "p90 head": dict(zip(names, np.nanpercentile(ph_st, 90, 0).tolist()))},
                  "rungs": {"median head": dict(zip(rungs, np.nanmedian(ph_err, 0).tolist())),
                            "p90 head": dict(zip(rungs, np.nanpercentile(ph_err, 90, 0).tolist()))},
                  "sigma_hat_U": float(np.nanmedian(S["sd_U"])), "cal_vs_actual_median": cva,
                  "full_oracle_max": fo}
    out = {"gate": gate, "files_sha": sig, "tasks": res}
    out_f.write_text(json.dumps(out, indent=1, default=float))
    return out

E1S = {}                        # at scale: {gate: {task: results}}
for g in (GATE, STRESS_GATE):
    r = scale_job(g)
    if r:
        E1S[g] = r["tasks"]

def rung(E, r, col="median head"):
    v = [E[t]["rungs"][col].get(r) for t in E if E[t]["rungs"].get(col)]
    v = [x for x in v if x is not None]
    return float(np.median(v)) if v else None

def _ms(E, rho, src):
    out = {}
    for t, r in E.items():
        m, s = r["rungs"]["median head"], r["summary"]["median head"]
        out[(src, rho, t)] = {"head_steps": r["head_steps"], "unread": s["unread"], "err_off": m["off"],
                              "err_cal": m["cal"], "removed": 1 - m["cal"] / m["off"],
                              "removed_abs": m["off"] - m["cal"], "mass_oracle": m["mass_oracle"],
                              "E_w": s["E_w"], "sd_U": s["sd_U"]}
    return out

ms = {}
for g, E in E1.items():
    ms.update(_ms(E, g, "collector"))
for g, E in E1S.items():
    ms.update(_ms(E, g, "at scale"))
MS = pd.DataFrame(ms).T.sort_index()
MS.index.names = ["source", "rho", "task"]
print("Matched states, median head per task. err_* = rel. L2 output error vs o_held; removed = 1 - err_cal/err_off")
print(MS.round(4).to_string())
MS.to_csv(RES / "matched_states.csv")
MED = {(s, g): MS.xs((s, g)).median() for s, g in {(i[0], i[1]) for i in MS.index}}
for s in ("collector", "at scale"):
    if (s, GATE) in MED and (s, STRESS_GATE) in MED:
        a, b = MED[(s, GATE)], MED[(s, STRESS_GATE)]
        print(f"\n{s}, median over tasks, rho {GATE:.2f} -> {STRESS_GATE:.2f}: unread share {a['unread']:.1%} -> "
              f"{b['unread']:.1%}; error without completion {a['err_off']:.4f} -> {b['err_off']:.4f}, with it "
              f"{a['err_cal']:.4f} -> {b['err_cal']:.4f}; removed {a['removed']:.0%} -> {b['removed']:.0%} "
              f"({a['removed_abs']:.4f} -> {b['removed_abs']:.4f}, {b['removed_abs'] / a['removed_abs']:.2f}x)")
if GATE in E1S:
    print(f"\nsigma_hat_U: collector {json.loads(SIGMA_FILE.read_text())['sigma_hat_U']:.3f} (the noise unit), "
          f"at scale {np.median([r['sigma_hat_U'] for r in E1S[GATE].values()]):.3f}")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#52514e",
                     "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e",
                     "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8,
                     "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})
BLUE, ORANGE, MUTED, BAND = "#2a78d6", "#eb6834", "#52514e", "#e6e5e1"
SH = json.loads(SIGMA_FILE.read_text()) if SIGMA_FILE.exists() else {}
SIG = SH.get("sigma_hat_U")
EC = E1S.get(GATE) or E1[GATE]          # matched-state curves: at scale when recorded, else the collector

def _ref(ax, x, label, side="left", y=1):
    ax.axvline(x, color=MUTED, ls="--", lw=1, zorder=1)
    ax.annotate(label, (x, y), xycoords=("data", "axes fraction"), textcoords="offset points",
                xytext=(-4 if side == "left" else 4, -4 if y else 4), ha="right" if side == "left" else "left",
                va="top" if y else "bottom", color=MUTED, fontsize=9)

def _errbar(ax, pts, **kw):
    pts = sorted(pts, key=lambda p: p[0])
    ax.errorbar([x for x, _ in pts], [r["delta"] for _, r in pts],
                yerr=[[r["delta"] - r["lo"] for _, r in pts], [r["hi"] - r["delta"] for _, r in pts]],
                color=BLUE, marker="o", ms=8, capsize=3, zorder=3, **kw)

fl = [abs(D[n]["delta"]) for n in ("cal_repeat", "cal_ep") if D.get(n)]
floor = max(fl) if fl else 0.0
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
# top left: common scale, end to end
ax = axes[0, 0]
ax.axhline(0, color=MUTED, lw=1, zorder=1)
ax.axhspan(-floor, floor, color=BAND, lw=0)
pts = [(0.0, dict(delta=0.0, lo=0.0, hi=0.0))] + [(np.log2(c), D[f"scale{c:g}"]) for c in (0.5, 0.8, 1.25, 2.0, 4.0)
                                                  if D.get(f"scale{c:g}")]
_errbar(ax, pts, lw=2)
if D.get("off"):
    x0 = min(x for x, _ in pts) - 1.2
    _errbar(ax, [(x0, D["off"])], lw=0, elinewidth=2)
    ax.annotate("off (c = 0)", (x0, D["off"]["delta"]), textcoords="offset points", xytext=(8, 0), va="center",
                color=MUTED)
if SH.get("c_hat"):
    _ref(ax, np.log2(SH["c_hat"]), f"measured fill error\nc = {SH['c_hat']:.2f}", y=0)
ax.set_xlabel("log2 c  (common scale on unread-window mass)")
ax.set_ylabel("Δ LongBench average vs cal (points)")
ax.set_title("End to end: common-scale mass error", loc="left")
# top right: redistribution, end to end, seeds
ax = axes[0, 1]
ax.axhline(0, color=MUTED, lw=1, zorder=1)
ax.axhspan(-floor, floor, color=BAND, lw=0)
pts = [(0, dict(delta=0.0, lo=0.0, hi=0.0))]
for m in NOISE_MULTS:
    r, per = SEEDS[m]
    sd = [p["delta"] for p in per.values() if p]
    ax.scatter([m] * len(sd), sd, s=64, facecolor="none", edgecolor=MUTED, lw=1.5, zorder=2)
    if r:
        pts.append((m, r))
_errbar(ax, pts, lw=2)
ax.annotate("open circles: single seeds; filled: seed mean, 95% CI", (0, 1), xycoords="axes fraction",
            textcoords="offset points", xytext=(4, -4), va="top", color=MUTED, fontsize=9)
ax.set_xticks([0, *NOISE_MULTS])
ax.set_xlabel(r"$\sigma\,/\,\hat{\sigma}_U$")
ax.set_title("End to end: redistribution, total preserved", loc="left")
# bottom: the same perturbations on matched states
ax = axes[1, 0]
rs = [("scale0.5", 0.5), ("scale0.8", 0.8), ("cal", 1.0), ("scale1.25", 1.25), ("scale2", 2.0), ("scale4", 4.0)]
ax.plot([np.log2(c) for _, c in rs], [rung(EC, r) for r, _ in rs], color=BLUE, lw=2, marker="o", ms=8, zorder=3)
for r, lab, dy in (("off", "off", 3), ("mass_oracle", "mass oracle", -3)):
    ax.axhline(rung(EC, r), color=MUTED, lw=1, ls=":", zorder=1)
    ax.annotate(lab, (0, rung(EC, r)), xycoords=("axes fraction", "data"), textcoords="offset points",
                xytext=(4, dy), va="bottom" if dy > 0 else "top", color=MUTED, fontsize=9)
if SH.get("c_hat"):
    _ref(ax, np.log2(SH["c_hat"]), f"c = {SH['c_hat']:.2f}")
ax.set_xlabel("log2 c")
ax.set_ylabel("output error vs o_held (median head, median task)")
ax.set_title("Matched states: common scale", loc="left")
ax.margins(y=0.12)
ax = axes[1, 1]
sg = [0.0, 0.25, 0.5, 1.0, 2.0]
ax.plot(sg, [rung(EC, r) for r in ("cal", "noise0.25", "noise0.5", "noise1", "noise2")], color=BLUE, lw=2,
        marker="o", ms=8, zorder=3)
if SIG:
    for m in NOISE_MULTS:
        if m * SIG <= 2.2:
            _ref(ax, m * SIG, f"{m}×" + r"$\hat{\sigma}_U$", side="right")
ax.set_xlabel(r"$\sigma$ (nats)")
ax.set_title("Matched states: redistribution", loc="left")
fig.text(0.01, -0.02, "Top: 95% bootstrap CIs, paired examples; grey band = largest |Δ| of the floors (cal_repeat, "
         f"cal_ep). Bottom: rel. L2 attention-output error vs reading every held window "
         f"({'recorder arms, all 50 prompts per task' if E1S.get(GATE) else 'collector'}).", color=MUTED, fontsize=9)
fig.tight_layout()
fig.savefig(RES / "sensitivity.png", dpi=200, bbox_inches="tight")
plt.show()

# stress: matched-state error vs rho (off, cal), and the end-to-end benefit of completion at each rho
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
src = "at scale" if all(("at scale", g) in MED for g in (GATE, STRESS_GATE)) else "collector"
if all((src, g) in MED for g in (GATE, STRESS_GATE)):
    for key, col, lab in (("err_off", ORANGE, "off"), ("err_cal", BLUE, "cal")):
        ys = [MED[(src, STRESS_GATE)][key], MED[(src, GATE)][key], 0.0]
        ax.plot([0, 1, 2], ys, color=col, lw=2, marker="o", ms=8, zorder=3, label=lab)
        ax.annotate(lab, (0, ys[0]), textcoords="offset points", xytext=(-10, 0), ha="right", va="center",
                    color=MUTED, fontsize=9)
    ax.set_xticks([0, 1, 2], [f"{STRESS_GATE:.2f}", f"{GATE:.2f}", "1.00"])
    ax.set_xlim(-0.7, 2.3)
    ax.legend(frameon=False, loc="upper right", fontsize=9)
ax.set_xlabel("read ratio ρ  (1.00: nothing unread, 0 by construction)")
ax.set_ylabel("output error vs o_held (median head, median task)")
ax.set_title(f"Matched states ({src}): what completion stands in for", loc="left")
ax = axes[1]
ax.axhline(0, color=MUTED, lw=1, zorder=1)
ax.axhspan(-floor, floor, color=BAND, lw=0)
pts = [(i, r) for i, r in ((0, B10), (1, B25)) if r]
if pts:
    _errbar(ax, pts, lw=0, elinewidth=2)
ax.set_xticks([0, 1], [f"{STRESS_GATE:.2f}", f"{GATE:.2f}"])
ax.set_xlim(-0.6, 1.6)
ax.set_xlabel("read ratio ρ")
ax.set_ylabel("cal − off, LongBench average (points)")
ax.set_title("End to end: benefit of completion", loc="left")
fig.tight_layout()
fig.savefig(RES / "stress.png", dpi=200, bbox_inches="tight")
plt.show()

## 9. Consolidated: attention-output error, accuracy and cost at the fixed KV-memory budget; summary; run status

Every row holds the same KV memory: budget 0.10, `quant_ratio` 0.70 in bytes mode, the same card widths and the same
prompts. The cell checks this from the sidecars (each row's resolved `retained_bytes` on each task's first batch) rather
than from the config. ρ and the completion term change what is **read** and how it is **combined**, not what is
**held**.

| Column | Kind | What it is |
|---|---|---|
| KV held | measured (resolved geometry) | retained K/V bytes per sequence, all layers, mean over tasks |
| KV read / step | arithmetic | (held − N_q·(1−ρ)·b_q) / held. Cards are read for every window; int2 data only for the fraction ρ that is opened |
| err vs held | measured, matched states | rel. L2 attention-output error vs reading every held window (median head, median task). From the recorder arms (all 50 prompts) where available, else the collector. 0 at ρ = 1.0 by construction; `~` marks noise levels interpolated between the σ rungs |
| LongBench | measured | task-average ROUGE-L on 50 examples per task |
| Δ vs cal | measured, paired | 95% bootstrap CI. `CI excl. 0` is the significance call. The CIs are not corrected for multiple comparisons |
| s / example | measured, batched | native arms only, all prefill and decode included. Epilogue arms (†) carry an evaluation-only torch epilogue |

**The summary keeps two claims apart.**
1. Experiment 1 measures how much of the read gate's *internal* approximation error the completion term removes, on
   identical decode states. It is precise.
2. The LongBench deltas measure whether that changes *task accuracy*. At 50 examples per task they resolve only changes
   larger than the printed MDE.

A precise (1) next to a non-significant (2) means the downstream effect is smaller than this N can detect. It is
**not** evidence of no effect, and it is not evidence that (1) carries over to accuracy. The summary is generated from the
numbers and written to `results/reviewer2_summary.md`.

Decode-time context (`DISTANCE_TO_GOAL.md` §2): measured TPOT at ρ = 1.0 ties with ρ = 0.25 to within 0.2–1.6%. The
"KV read" column is about memory traffic and makes no latency claim.

In [ ]:
probe = r"""
import json, sys
from modules.quant.quantizer import bytes_per_q_window
from modules.quant.sketch import CARD_BITS_DEFAULT as B, card_field_layout, sketch_bytes_per_head
c = json.load(open(sys.argv[1])); ws = int(sys.argv[2])
H = c["num_key_value_heads"]; D = c.get("head_dim") or c["hidden_size"] // c["num_attention_heads"]
print(json.dumps({"layers": c["num_hidden_layers"], "H_kv": H, "D": D, "ws": ws,
                  "b_qdata": bytes_per_q_window(H, D, ws), "b_card": H * sketch_bytes_per_head(D, ws, B),
                  "b_vbar": H * (card_field_layout(D, B.vm)[0] + 2)}))
"""
# Window prices from the modules that define the formats (per layer, all KV heads), not re-derived here.
GEOM = json.loads(subprocess.check_output([PY, "-c", probe, str(Path(MODEL_DIR) / "config.json"),
                                           str(OPERATING_POINT["window_size"])], cwd=REPO, env=ENV, text=True))
GKEYS = ("prefill_len", "retained_bytes", "N_q", "top_k_fp", "bytes_per_gate_card", "total_budget_bytes")

def _geom(n, t):
    return (META.get((n, t)) or {}).get("resolved_geometry_first_example")

def held_read(names, rho):
    """(KV held in MB per sequence over all layers, arithmetic read share per step, same memory as cal?)"""
    held, share, same = [], [], True
    for t in TASKS:
        gs, gc = [g for g in (_geom(n, t) for n in names) if g], _geom("cal", t)
        if not gs or not gc:
            return None, None, None
        same &= all(all(g.get(k) == gc.get(k) for k in GKEYS) for g in gs)
        Rb = gs[0]["retained_bytes"]
        held.append(Rb * GEOM["layers"] / 1e6)
        share.append((Rb - gs[0]["N_q"] * (1 - rho) * GEOM["b_qdata"]) / Rb)
    return float(np.mean(held)), float(np.mean(share)), same

def realised(names):
    v = [META[(n, t)]["read_gate"]["read_fraction"] for n in names for t in TASKS if (n, t) in META]
    return "—" if not v else (f"{min(v):.3f}" if max(v) - min(v) < 5e-4 else f"{min(v):.3f}-{max(v):.3f}")

def sec_per_ex(names):
    if any(ARMS[n]["comp"]["mode"] != "native" for n in names):
        return "†"
    v = [1.0 / META[(names[0], t)]["examples_per_second"] for t in TASKS
         if META.get((names[0], t), {}).get("examples_per_second")]
    return f"{np.mean(v):.2f}" if len(v) == len(TASKS) else "—"

def noise_rung(sig, E):
    xs, rs = [0.0, 0.25, 0.5, 1.0, 2.0], ["cal", "noise0.25", "noise0.5", "noise1", "noise2"]
    ys = [rung(E, r) for r in rs]
    if sig is None or any(y is None for y in ys) or sig > xs[-1]:
        return None
    return float(np.interp(sig, xs, ys))

ES = {g: (E1S.get(g) or E1.get(g) or {}) for g in (GATE, STRESS_GATE)}   # matched states: at scale, else collector
ROWS = [(CEIL, [CEIL], 1.0, "All windows read"),
        ("cal", ["cal"], GATE, "Calibrated completion (QEvict)"),
        ("off", ["off"], GATE, "No completion"),
        ("uncal", ["uncal"], GATE, "Uncalibrated completion"),
        (SC, [SC], STRESS_GATE, "Calibrated completion"),
        (SO, [SO], STRESS_GATE, "No completion"),
        *[(f"scale{c:g}", [f"scale{c:g}"], GATE, f"Calibrated, unread masses x {c:g}") for c in (0.5, 0.8, 1.25, 2.0, 4.0)],
        *[(f"noise{m}x", [f"noise{m}x_s{s}" for s in NOISE_SEEDS], GATE, f"Calibrated, redistribution {m}x sigma_hat_U")
          for m in NOISE_MULTS],
        *[(f"coupled_scale{c:g}", [f"coupled_scale{c:g}"], GATE, f"Coupled scores, masses x {c:g}") for c in (0.5, 2.0)]]
MS_OF = {CEIL: 0.0, "cal": rung(ES[GATE], "cal"), "off": rung(ES[GATE], "off"), "uncal": rung(ES[GATE], "uncal"),
         SC: rung(ES[STRESS_GATE], "cal"), SO: rung(ES[STRESS_GATE], "off"),
         **{f"scale{c:g}": rung(ES[GATE], f"scale{c:g}") for c in (0.5, 0.8, 1.25, 2.0, 4.0)},
         **{f"noise{m}x": noise_rung(None if SIG is None else m * SIG, ES[GATE]) for m in NOISE_MULTS}}

cons, memory_ok = [], True
for key, names, rho, label in ROWS:
    names = [n for n in names if have(n)]
    if not names:
        continue
    if key.startswith("noise"):
        dl = SEEDS[int(key[5:-1])][0]
        label += f", mean of {len(names)} seeds" if len(names) > 1 else f", seed {names[0].rsplit('_s', 1)[1]}"
    else:
        dl = None if key == "cal" else D.get(key)
    held, share, same = held_read(names, rho)
    memory_ok &= bool(same)
    e = MS_OF.get(key)
    cons.append({"configuration": label, "arm": key, "rho": f"{rho:.2f}", "realised rho": realised(names),
                 "KV held (MB)": "—" if held is None else f"{held:.1f}" + ("" if same else " (differs!)"),
                 "KV read/step": "—" if share is None else f"{share:.1%}",
                 "err vs held": "—" if e is None else ("0 (by constr.)" if key == CEIL else
                                                       ("~" if key.startswith("noise") else "") + f"{e:.4f}"),
                 "LongBench": f"{np.mean([task_score(n, t) for n in names for t in TASKS]):.2f}",
                 "Δ vs cal [95% CI]": "—" if key == "cal" else ci(dl),
                 "CI excl. 0": "" if dl is None else ("yes" if dl["sig"] else "no"),
                 "MDE80": "" if dl is None else f"{dl['mde80']:.2f}", "s/example": sec_per_ex(names)})
CONS = pd.DataFrame(cons).set_index("configuration")
print(CONS.to_string())
CONS.to_csv(RES / "consolidated.csv")
card_sc = (_geom("cal", TASKS[0]) or {}).get("bytes_per_gate_card")
if card_sc != GEOM["b_card"]:
    print(f"WARNING: the sidecar's card price ({card_sc} B) is not the repo's ({GEOM['b_card']} B)")
print("\nFixed KV memory: " + ("every row holds exactly cal's retained bytes on every task (checked from the sidecars)."
                               if memory_ok else "NOT every row holds cal's bytes: see 'differs!' above."))

# ---------- the Reviewer-2 summary, generated from the numbers above ----------
def _red(E):
    return {t: 1 - E[t]["rungs"]["median head"]["cal"] / E[t]["rungs"]["median head"]["off"] for t in E}
L_ = [f"# Reviewer 2: completion accuracy, sensitivity, and the quality-memory trade-off",
      f"Commit `{SHA[:10]}`; Llama-3.1-8B-Instruct; budget {CACHE_BUDGET}, quant_ratio {OPERATING_POINT['quant_ratio']} "
      f"(bytes); LongBench {', '.join(TASKS)}, {NUM_SAMPLES} examples each, prompts of "
      + ", ".join(f"{PROMPT_TOKENS[t]}" for t in TASKS) + " tokens; batched. Generated by section 9.", ""]
L_ += ["## 1. Internal approximation error (Experiment 1: measured, matched decode states)"]
for g in (GATE, STRESS_GATE):
    for src, E in (("all 50 templated prompts per task, recorded inside the batched runs", E1S.get(g)),
                   (f"collector, {EXP1_PROMPTS} documents per task", E1.get(g))):
        if not E:
            continue
        r = _red(E)
        L_ += [f"* rho = {g:.2f} ({src}): calibrated completion removes **{np.median(list(r.values())):.0%}** of the "
               f"read gate's attention-output error (median over tasks; " + ", ".join(f"{t} {v:.0%}" for t, v in r.items())
               + f"; {sum(E[t]['head_steps'] for t in E):,} head-steps). Error without completion "
               f"{rung(E, 'off'):.4f}, with it {rung(E, 'cal'):.4f}; a perfect mass estimate reaches "
               f"{rung(E, 'mass_oracle'):.4f}."]
if PAPER_ERROR_REDUCTION is not None and (E1S.get(GATE) or E1.get(GATE)):
    E = E1S.get(GATE) or E1[GATE]
    cands = {"median of per-task reductions": float(np.median(list(_red(E).values()))),
             "mean of per-task reductions": float(np.mean(list(_red(E).values()))),
             "1 - median err(cal) / median err(off)": 1 - rung(E, "cal") / rung(E, "off")}
    hit = [k for k, v in cands.items() if abs(v - PAPER_ERROR_REDUCTION) <= 0.01]
    L_ += [f"* Check against the quoted {PAPER_ERROR_REDUCTION:.0%}: " + (
        f"reproduced as the {hit[0]}." if hit else "NOT reproduced (" + ", ".join(
            f"{k} = {v:.1%}" for k, v in cands.items()) + "). The quoted figure came from a different run (budget, "
        "tasks or prompts): quote this run's.")]
L_ += ["* This is a property of the approximation on identical states. Its precision does not depend on the LongBench N.", ""]
L_ += ["## 2. Downstream accuracy (LongBench: measured, paired, 95% bootstrap CI)"]
if B25:
    L_ += [f"* Completion vs none at rho = {GATE:.2f}: cal - off = **{ci(B25)}** points on the task average, "
           f"{NUM_SAMPLES} examples per task. " + (
               "The CI includes 0: **not statistically significant** at this N." if not B25["sig"] else
               "The CI excludes 0: significant at this N.")
           + f" The smallest change this N detects with 80% power is {B25['mde80']:.2f} points."]
    if not B25["sig"]:
        L_ += ["* Read together: the internal error reduction is large and precisely measured; the downstream "
               "difference is below what this N resolves. The data support *completion makes the approximation "
               "substantially more accurate*. They do not yet support *completion changes LongBench accuracy*, and "
               "they do not show that it doesn't."]
    else:
        L_ += ["* Read together: the internal error reduction and a significant downstream difference " + (
            "point the same way: completion improves task accuracy at this N." if B25["delta"] > 0 else
            "point opposite ways: completion LOWERS task accuracy despite the smaller internal error. Report it as "
            "found; it needs explaining before the claim is made.")]
L_ += [""]
L_ += [f"## 3. rho = 1.00 (every window read) vs calibrated completion at rho = {GATE:.2f}"]
if have(CEIL):
    r1 = delta("cal", CEIL)
    L_ += [f"* cal - rho 1.00 = {ci(r1)} points (" + ("CI excludes 0" if r1["sig"] else "CI includes 0") + ").",
           "* Share of the read-gate gap closed by completion: " + (
               f"{GAP['point']:.2f} [{GAP['lo']:.2f}, {GAP['hi']:.2f}]." if GAP and GAP["den_lo"] > 0 else
               "undefined: rho = 1.00 does not beat off significantly at this N.")]
L_ += [""]
L_ += [f"## 4. rho = {STRESS_GATE:.2f} stress test"]
for s in ("at scale", "collector"):
    if all((s, g) in MED for g in (GATE, STRESS_GATE)):
        a, b = MED[(s, GATE)], MED[(s, STRESS_GATE)]
        L_ += [f"* Matched states ({s}): unread share of held attention {a['unread']:.1%} -> {b['unread']:.1%}; output "
               f"error without completion {a['err_off']:.4f} -> {b['err_off']:.4f}, with it {a['err_cal']:.4f} -> "
               f"{b['err_cal']:.4f}; error removed {a['removed_abs']:.4f} -> {b['removed_abs']:.4f} "
               f"({b['removed_abs'] / a['removed_abs']:.2f}x), {a['removed']:.0%} -> {b['removed']:.0%} of it."]
        break
if B10:
    L_ += [f"* End to end: cal - off = {ci(B10)} at rho = {STRESS_GATE:.2f} vs {ci(B25)} at rho = {GATE:.2f}; "
           f"difference-in-differences {ci(DID)} (" + ("CI excludes 0" if DID and DID["sig"] else
                                                       "CI includes 0: this N cannot tell") + ")."]
L_ += ["", "## 5. End-to-end sensitivity (Δ vs cal, points [95% CI]; * = CI excludes 0)"]
sens = []
for c in (0.5, 0.8, 1.25, 2.0, 4.0):
    r = D.get(f"scale{c:g}")
    if r:
        sens.append(r)
        L_ += [f"* masses x {c:g}: {ci(r)}" + (" *" if r["sig"] else "")]
for m in NOISE_MULTS:
    r, per = SEEDS[m]
    if r:
        sens.append(r)
        L_ += [f"* noise {m}x sigma_hat_U, " + (f"mean of {len(per)} seeds" if len(per) > 1 else "one seed")
               + f": {ci(r)}" + (" *" if r["sig"] else "")]
for c in (0.5, 2.0):
    r = D.get(f"coupled_scale{c:g}")
    if r:
        sens.append(r)
        L_ += [f"* coupled scores, masses x {c:g}: {ci(r)}" + (" *" if r["sig"] else "")]
fls = ", ".join(f"{n} {D[n]['delta']:+.2f}" for n in ("cal_repeat", "cal_ep") if D.get(n))
L_ += [f"* {sum(r['sig'] for r in sens)} of {len(sens)} points have a CI excluding 0; unadjusted, about "
       f"{0.05 * len(sens):.1f} would by chance. Floors: {fls}.", ""]
L_ += ["## 6. Cost at the fixed KV-memory budget"]
hc = held_read(["cal"], GATE)[0]
L_ += [f"* Every row holds the same KV memory ({'?' if hc is None else f'{hc:.1f}'} MB per sequence over "
       f"{GEOM['layers']} layers, mean over tasks)" + ("; checked from the sidecars." if memory_ok else
                                                      "; NOT confirmed by the sidecars.")]
for key, rho in ((CEIL, 1.0), ("cal", GATE), (SC, STRESS_GATE)):
    if have(key) and held_read([key], rho)[1] is not None:
        L_ += [f"* rho = {rho:.2f}: reads {held_read([key], rho)[1]:.1%} of the held bytes per decode step "
               f"(arithmetic); measured {sec_per_ex([key])} s/example, batched."]
L_ += [f"* The completion term's only resident cost is the value centroid on each card: {GEOM['b_vbar']} B per int2 "
       f"window across {GEOM['H_kv']} KV heads, {GEOM['b_vbar'] / (GEOM['b_qdata'] + GEOM['b_card']):.1%} of the "
       f"{GEOM['b_qdata'] + GEOM['b_card']} B the window costs.", ""]
L_ += ["## Caveats",
       f"* Prompts are middle-truncated to a fixed length per task ({', '.join(f'{t} {PROMPT_TOKENS[t]}' for t in TASKS)} "
       "tokens) so that rows batch without padding; the first 50 examples at least that long are used. Every arm sees "
       "the same prompts.",
       f"* The collector's prompts are document text, not the templated prompt; the recorder arms' are the real ones.",
       "* CIs are paired bootstraps over examples, tasks resampled independently, not corrected for multiple comparisons."]
SUMMARY = "\n".join(L_)
(RES / "reviewer2_summary.md").write_text(SUMMARY + "\n")
md = ["| " + " | ".join(["Configuration", *CONS.columns]) + " |", "|---" * (len(CONS.columns) + 1) + "|"]
md += ["| " + " | ".join([str(i), *[str(v) for v in r]]) + " |" for i, r in CONS.iterrows()]
(RES / "rebuttal_table.md").write_text("\n".join(md) + f"\n\ncommit {SHA}\n")
print("\n" + SUMMARY)
status_table()
print(f"\nAll results: {RES}   commit {SHA[:10]}")